# 01 — Local and Hosted Model Gateway

**First Finance - Arnaud Demes**

Build the first working layer of the Financial Analyst Copilot: one model call that can run locally with Ollama or through OpenAI without changing the lesson code.

## Learning objectives

By the end of this notebook, you can:

- explain the message contract of a chat model;
- distinguish a model from the application built around it;
- configure Ollama or OpenAI through the same Python boundary;
- measure model latency and retain run metadata;
- expose why a successful API call can still produce a poor financial answer; and
- identify which model choices belong in configuration rather than notebook code.

## Before you start

**Estimated time:** 30 minutes — about 10 minutes of explanation and 20 minutes of guided execution.

Work from the repository root and use one provider:

- **Ollama:** Ollama is running and `qwen3:8b` is available locally.
- **OpenAI:** `OPENAI_API_KEY` is already defined in your shell. Never paste the key into this notebook.

Check your environment before opening Jupyter:

```bash
uv run python scripts/setup_check.py --provider ollama
```

or:

```bash
uv run python scripts/setup_check.py --provider openai
```

> **Expected result:** the setup check prints `PASS` for Python, core imports and the selected provider. If it does not, use the troubleshooting section near the end before continuing.

## How to use this notebook

1. Run the cells from top to bottom.
2. Read each **Expected result** note before running the next cell.
3. Do not optimise the prompt during the failure lab; the weak result is part of the lesson.
4. Complete the comparison table in the challenge after the guided section.

The notebook uses a recorded response only when the automated test harness sets `FINAI_LIVE_MODE=0`. In a normal student session it calls the provider you selected.

## Where this fits

This is the first executable layer of the capstone. Today the model receives a question and returns measured text. The next notebook turns that text into a validated financial object. The rest of the course adds context decisions, retrieval, document engineering, evaluation, workflows, agents and MCP.

```text
Question → Settings → Model gateway → Response + latency + tokens
                                      ↓
        structured output → RAG → evaluation → agents → MCP
```

> **Important:** a language model is a component. The Financial Analyst Copilot is the complete system that controls its context, tools, outputs, evidence and evaluation.

## One configuration contract

Every LLM-dependent notebook reads the same environment variables:

| Variable | Local example | Hosted example |
|---|---|---|
| `FINAI_MODEL_PROVIDER` | `ollama` | `openai` |
| `FINAI_CHAT_MODEL` | `qwen3:8b` | `gpt-5-mini` |
| `FINAI_EMBEDDING_PROVIDER` | `ollama` | `openai` |
| `FINAI_EMBEDDING_MODEL` | `qwen3-embedding:0.6b` | `text-embedding-3-small` |

The provider adapters change. The lesson logic does not. API keys stay in the environment and never appear in notebook cells.

In [ ]:
from __future__ import annotations

import os
from time import perf_counter
from typing import Any

from finai_academy.lesson_support import RecordedChatModel, evaluate_grounding
from finai_academy.providers import (
    ModelRun,
    create_chat_model,
    normalize_token_usage,
    provider_summary,
)
from finai_academy.settings import Settings

### Live mode and test mode

When you open the notebook normally, `FINAI_LIVE_MODE` defaults to `1` and the configured provider is called. The automated course suite sets it to `0` and uses a deterministic recorded response. That makes regression tests fast and free while preserving separate live Ollama and OpenAI acceptance runs.

The recorded response is a test double, not a claim that a provider was contacted. Its implementation lives in `finai_academy.lesson_support` so the learner-facing notebook stays focused on the gateway contract.

In [ ]:
OFFLINE_MODEL_NAME = "recorded-response-v2"

In [ ]:
settings = Settings.from_environment()
live_mode = os.getenv("FINAI_LIVE_MODE", "1") == "1"
model = create_chat_model(settings) if live_mode else RecordedChatModel()

print("Execution mode:", "live" if live_mode else "offline fixture")
print("Provider configuration:", provider_summary(settings))

> **Expected result:** `Execution mode` is `live` in class and `Provider configuration` shows the selected provider and model. The diagnostic must never print an API key. If you see `offline fixture`, you are running the deterministic test path rather than a live provider.

## The chat message contract

A chat call is an ordered list of messages, not one magical prompt string. The two roles used here have different responsibilities:

- **system** — stable application instructions and boundaries;
- **human** — the current request and its data.

Later we will add tool messages and conversation state. Keeping roles explicit makes prompts easier to inspect, test and version.

In [ ]:
first_messages = [
    ("system", "You are a careful financial analyst assistant."),
    ("human", "In no more than five sentences, what is happening with AI demand?"),
]
first_messages

## Measure the call

Latency, provider, model and token usage are application data. Capturing them at the gateway gives later notebooks a consistent record that Lesson 07 can log and compare. MLflow is deliberately not needed yet.

> **Run:** execute the function cell and then the first-call cell. **Expected result:** a non-empty answer followed by provider, model, latency and token usage when the provider returns it. A first local call may be slower because Ollama has to load the model into memory.

In [ ]:
def invoke_with_metrics(
    chat_model: Any,
    messages: list[tuple[str, str]],
    configured_settings: Settings,
) -> tuple[ModelRun, Any]:
    started = perf_counter()
    response = chat_model.invoke(messages)
    latency_ms = (perf_counter() - started) * 1_000
    run = ModelRun(
        provider=configured_settings.provider if live_mode else "offline",
        model=configured_settings.chat_model if live_mode else OFFLINE_MODEL_NAME,
        text=str(response.content),
        latency_ms=latency_ms,
        token_usage=normalize_token_usage(getattr(response, "usage_metadata", None)),
    )
    return run, response

In [ ]:
first_run, first_response = invoke_with_metrics(model, first_messages, settings)
print(first_run.text)
print(f"\nprovider={first_run.provider} model={first_run.model} latency={first_run.latency_ms:.0f} ms")
if first_run.token_usage is None:
    print("Token usage: unavailable for this provider response")
else:
    print(
        "Token usage:",
        f"input={first_run.token_usage.input_tokens}",
        f"output={first_run.token_usage.output_tokens}",
        f"total={first_run.token_usage.total_tokens}",
    )

### Inspect more than the text

Provider SDKs often return finish reasons, token usage, model identifiers and safety metadata beside the answer. The exact fields vary, which is why the application normalizes only the metadata it truly needs.

> **Observe:** the metadata dictionary is provider-specific. Application code should not assume that every provider returns the same keys.

In [ ]:
{
    "normalized_token_usage": first_run.token_usage,
    "provider_metadata": getattr(first_response, "response_metadata", {}),
}

### Streaming changes delivery, not answer quality

Streaming displays partial model output as it arrives. It can improve perceived responsiveness, but it does not add evidence, improve factuality or change the final answer contract. The application must still collect the complete text before validation.

> **Expected result:** text appears after `Streaming demo:` and `streamed_text` contains the complete response. The offline path replays the recorded answer; the live path uses the selected provider's stream.

In [ ]:
print("Streaming demo:", end=" ", flush=True)
if live_mode:
    streamed_parts = []
    for chunk in model.stream(first_messages):
        part = str(chunk.content)
        streamed_parts.append(part)
        print(part, end="", flush=True)
    streamed_text = "".join(streamed_parts)
else:
    streamed_parts = first_run.text.split()
    for part in streamed_parts:
        print(part, end=" ", flush=True)
    streamed_text = " ".join(streamed_parts)
print()
assert streamed_text.strip(), "The streamed response must not be empty."

## What the main parameters control

- **Model** changes capability, latency, price and local hardware requirements.
- **Temperature** changes sampling behaviour; lower is useful for repeatable analytical tasks but does not guarantee factuality.
- **Maximum output tokens** limits generation length, not the amount of evidence the model can read.
- **Context window** is shared by instructions, conversation, documents, tool results and the generated answer.

Different model families expose different sampling controls. The gateway should not force a parameter that the selected provider or model does not support.

## Failure lab

The first call can return fluent text, yet the request is impossible to answer professionally. It omitted:

1. the company;
2. the reporting period;
3. the source material;
4. the evidence standard; and
5. the required output.

A model cannot infer an application contract reliably from `In no more than five sentences, what is happening with AI demand?`. The length limit keeps the demonstration usable, but it does not define a professional analyst output. The problem is not solved by selecting a larger model.

## Improve the request before adding infrastructure

We still do not have RAG. We can nevertheless make the task explicit and provide a compact evidence card from a real filing. Notice how instructions and source data remain separated and how every fact receives a stable identifier.

> **Expected result:** the answer identifies NVIDIA and fiscal 2026, uses at least two supplied metrics, cites fact identifiers such as `[F1]`, and states something the evidence cannot establish. Wording will vary by model.

In [ ]:
NVIDIA_SOURCE_URL = (
    "https://www.sec.gov/Archives/edgar/data/1045810/"
    "000104581026000021/nvda-20260125.htm"
)
evidence_card = {
    "F1": "NVIDIA fiscal 2026 revenue was $215.9 billion, up 65% year on year.",
    "F2": "Data Center revenue was $193.7 billion, up 68% year on year.",
    "F3": "Gaming revenue was $16.0 billion, up 41% year on year.",
    "F4": (
        "Gross margin decreased; the filing says it was also affected by a $4.5 billion "
        "H20 excess-inventory and purchase-obligation charge."
    ),
}
evidence_text = "\n".join(f"[{fact_id}] {fact}" for fact_id, fact in evidence_card.items())

grounded_messages = [
    (
        "system",
        ("You are a careful equity-research assistant. Use only the supplied evidence card. "
        "Cite the relevant fact identifier in square brackets after every factual claim. "
        "Do not calculate a new ratio or introduce a number or year absent from the card. "
        "Do not provide a recommendation, valuation or price target."),
    ),
    (
        "human",
        ("Return only three concise bullets labelled Growth, Business lines and Limitation. "
        "Start the Growth bullet with the words 'NVIDIA fiscal 2026' so the answer is self-contained. "
        "For Business lines, report the F2 and F3 values separately; do not calculate or compare shares. "
        "Use at least two metrics and at least two fact citations. Use this exact final bullet: "
        "'- Limitation: The evidence does not establish valuation, a price target or future guidance.' "
        "Do not add any text after the Limitation bullet.\n\n"
        f"<evidence_card>\n{evidence_text}\n</evidence_card>"),
    ),
]

In [ ]:
grounded_run, _ = invoke_with_metrics(model, grounded_messages, settings)
print(grounded_run.text)
print(f"\nlatency={grounded_run.latency_ms:.0f} ms")

### Check grounding before trusting fluency

The four checks below are deliberately transparent. They do not use another model and they do not prove investment quality. They confirm that this guided answer identifies the company and period, uses only supplied numbers, cites the evidence, and keeps one explicit limitation.

The official source is NVIDIA's fiscal 2026 Form 10-K filed with the U.S. Securities and Exchange Commission: [SEC filing](https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm).

In [ ]:
grounding_result = evaluate_grounding(grounded_run.text)
for criterion, passed in grounding_result.checks.items():
    print(f"{'PASS' if passed else 'REVIEW':6} {criterion}")
print(f"Grounding score: {grounding_result.score}/{grounding_result.maximum}")

### Debrief

The improved call uses real, labelled evidence and passes a small observable rubric. It is still only free-form text: we have not validated its shape, checked every possible claim, or retrieved from a long document. That is intentional because each limitation creates the next lesson.

The engineering improvement in this notebook is the **provider boundary**. The prompt improvement previews Notebook 02.

## Switch providers without changing code

Start Jupyter with one of these configurations and run the notebook from the top:

```bash
FINAI_MODEL_PROVIDER=ollama FINAI_CHAT_MODEL=qwen3:8b uv run --extra ai jupyter lab
```

```bash
FINAI_MODEL_PROVIDER=openai FINAI_CHAT_MODEL=gpt-5-mini OPENAI_API_KEY=... uv run --extra ai jupyter lab
```

For the capstone demonstration, `FINAI_CHAT_MODEL=gpt-5.1` is a configuration change, not a code change.

## Verification

The assertions check the deterministic gateway contract. The grounding rubric remains visible but non-fatal because model wording is non-deterministic: a result below 4/4 should produce `REVIEW`, not crash the notebook. Later evaluation notebooks add dataset-level quality gates.

> **Target result:** `Grounding score: 4/4`. The final line must always be `PASS — provider-neutral model gateway verified` when the provider boundary executed correctly.

In [ ]:
assert first_run.text.strip(), "The model response must not be empty."
assert grounded_run.text.strip(), "The grounded response must not be empty."
assert first_run.latency_ms >= 0
assert grounded_run.latency_ms >= 0
assert settings.provider in {"ollama", "openai"}
print(
    "PASS — guided grounding target reached"
    if grounding_result.passed
    else "REVIEW — improve the answer against the visible grounding criteria"
)
print("PASS — provider-neutral model gateway verified")

## Knowledge check

Answer before expanding the solutions.

1. If streaming feels faster, has the factual quality improved?
2. Why can token usage be `None` even when the call succeeded?
3. Which change should require no notebook-code edit: Ollama to OpenAI, or free text to a validated schema?

<details><summary>Answers</summary>

1. No. Streaming changes delivery only; grounding and validation still determine quality.
2. Provider metadata is not uniform. The gateway retains `None` instead of inventing zero tokens.
3. Ollama to OpenAI is a configuration change behind the gateway. A validated schema is a new application contract and is built in Lesson 02.

</details>

## Challenge

Run the notebook once with Ollama and once with OpenAI. Record:

- model identifier;
- latency for each call;
- one material difference in the grounded answer;
- one privacy or cost trade-off; and
- whether that difference justifies provider-specific application code.

Use this table as your run log:

| Observation | Ollama | OpenAI |
|---|---|---|
| Model identifier |  |  |
| First-call latency |  |  |
| Grounded-call latency |  |  |
| Grounding score |  |  |
| Material answer difference |  |  |
| Privacy or cost trade-off |  |  |

Your conclusion should normally preserve the shared boundary even when one provider performs better.

## Troubleshooting

| Symptom | Likely cause | Action |
|---|---|---|
| `ModuleNotFoundError: finai_academy` | Jupyter was not started from the course environment | Stop Jupyter and run `uv run --extra ai jupyter lab` from the repository root. |
| Cannot connect to Ollama | The Ollama service is not running | Start Ollama, then rerun `scripts/setup_check.py --provider ollama`. |
| Ollama model not found | The configured model is missing | Run `ollama pull qwen3:8b`, then repeat the setup check. |
| OpenAI authentication error | The key is missing or invalid | Define `OPENAI_API_KEY` in the shell that starts Jupyter; do not place it in the notebook. |
| First local call is very slow | The model is loading into memory | Wait for the first call, then compare it with the second-call latency. |
| Output says `offline fixture` | `FINAI_LIVE_MODE=0` is set | Remove that environment variable for a live student run. |

After changing an environment variable, restart the kernel and run all cells from the top.

## Student checklist

Before moving to Notebook 02, confirm that you can check every item:

- [ ] I can identify the active provider and model without exposing a secret.
- [ ] I can explain the difference between a system message and a human message.
- [ ] I captured a non-empty response, latency and available token usage.
- [ ] I can explain why streaming changes delivery but not factual quality.
- [ ] I can explain why the vague question fails as an analyst request.
- [ ] I can trace an NVIDIA statement to a labelled fact in the SEC evidence card.
- [ ] The grounded answer reaches `4/4` on the visible rubric.
- [ ] I can switch provider through configuration without editing the lesson code.
- [ ] The verification cell prints `PASS`.

## Capstone integration

The capstone now owns five reusable elements:

1. `Settings.from_environment()` for provider-specific defaults;
2. `create_chat_model(settings)` for lazy provider construction;
3. `provider_summary(settings)` for safe diagnostics; and
4. `ModelRun` for normalized latency and token metadata; and
5. labelled evidence plus a transparent grounding check for the first real-company answer.

Notebook 02 will replace free-form text with a Pydantic-validated `AnalystBrief`.

## Recap

- A model call is an ordered message exchange.
- A model is only one component of an AI application.
- Provider choice belongs behind a small application boundary.
- Latency and available token counts are application data, not console trivia.
- Streaming changes delivery, not evidence or factual quality.
- Ollama and OpenAI can run the same notebook code.
- Real, labelled filing evidence makes a financial answer inspectable.
- A transparent rubric can separate execution success from basic grounding.
- Fluent text is still not a reliable financial product.
- The next engineering need is a validated output contract.